In [17]:
import os
import json
import numpy as np
from typing import List, Tuple
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
def bow_from_scratch(corpus: List[str]) -> Tuple[List[str], np.ndarray]:
    """Generates vocabulary and Bag-of-Words matrix from scratch."""
    vocab = sorted(list(set(w.lower() for doc in corpus for w in doc.split())))
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    
    matrix = np.zeros((len(corpus), len(vocab)), dtype=int)
    
    for row_idx, doc in enumerate(corpus):
        for word in doc.split():
            w_clean = word.lower()
            
            if w_clean in word_to_idx:
                matrix[row_idx, word_to_idx[w_clean]] += 1
    
    return vocab, matrix

In [19]:
def tfidf_from_scratch(corpus: List[str]) -> Tuple[List[str], np.ndarray]:
    """Generates normalized TF-IDF feature matrix from scratch."""
    
    vocab, bow = bow_from_scratch(corpus)
    N = len(corpus)
    
    # Term Frequency (TF)
    doc_lengths = bow.sum(axis=1, keepdims=True)
    tf = bow / np.maximum(doc_lengths, 1)
    
    # Inverse Document Frequency (IDF)
    df = (bow > 0).sum(axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1
    
    tfidf = tf * idf
    
    # L2 Normalization
    norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
    tfidf_normalized = np.where(
        norms > 0,
        tfidf / norms,
        tfidf
    )
    
    return vocab, tfidf_normalized

In [ ]:
def tfidf_from_scratch(corpus: List[str]) -> Tuple[List[str], np.ndarray]:
    """Generates normalized TF-IDF feature matrix from scratch."""
    
    vocab, bow = bow_from_scratch(corpus)
    N = len(corpus)
    
    # Term Frequency (TF)
    doc_lengths = bow.sum(axis=1, keepdims=True)
    tf = bow / np.maximum(doc_lengths, 1)
    
    # Inverse Document Frequency (IDF)
    df = (bow > 0).sum(axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1
    
    tfidf = tf * idf
    
    # L2 Normalization
    norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
    tfidf_normalized = np.where(
        norms > 0,
        tfidf / norms,
        tfidf
    )
    
    return vocab, tfidf_normalized

In [22]:
def run_plagiarism_check(
    corpus: List[str],
    doc_names: List[str],
    threshold: float = 0.70
):
    """Calculates cosine similarity across document pairs to flag potential plagiarism."""
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus).toarray()
    sim_matrix = cosine_similarity(tfidf_matrix)

    print(
        f"{'Document A':<20} | "
        f"{'Document B':<20} | "
        f"{'Cosine Sim':<12} | "
        f"{'Status':<15}"
    )
    
    print("-" * 75)

    for i in range(len(doc_names)):
        for j in range(i + 1, len(doc_names)):
            
            score = sim_matrix[i, j]
            
            status = (
                "FLAGGED PLAGIARISM"
                if score >= threshold
                else "CLEAN"
            )
            
            print(
                f"{doc_names[i]:<20} | "
                f"{doc_names[j]:<20} | "
                f"{score:.4f}       | "
                f"{status:<15}"
            )

In [23]:
# Dataset path for Jupyter Notebook
data_path = os.path.join(os.getcwd(), "academic_submissions.json")

print("Dataset path:", data_path)

Dataset path: d:\Github Desktop\Natural-Language-Processing-Laboratory\Assignment-3\academic_submissions.json


In [25]:
with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Dataset loaded successfully!")
print("Number of documents:", len(data))

Dataset loaded successfully!
Number of documents: 3


In [26]:
corpus = [item["content"] for item in data]
doc_ids = [item["id"] for item in data]

print("Corpus created successfully!")
print("Number of documents:", len(corpus))
print("Document IDs:", doc_ids)

Corpus created successfully!
Number of documents: 3
Document IDs: ['paper_1.txt', 'submission_A.txt', 'paper_2.txt']


In [27]:
print("--- Academic Submissions ---")

for item in data:
    print("\nDocument ID:", item["id"])
    print("Content:", item["content"])

--- Academic Submissions ---

Document ID: paper_1.txt
Content: Deep learning architectures have revolutionized natural language processing and image recognition tasks using convolutional neural networks.

Document ID: submission_A.txt
Content: Deep learning architectures have revolutionized natural language processing tasks using neural networks and convolutional layers.

Document ID: paper_2.txt
Content: Quantum computing utilizes quantum mechanics principles such as superposition and entanglement to perform complex matrix calculations.


In [28]:
# 1. From-Scratch Verification

vocab_scratch, tfidf_scratch = tfidf_from_scratch(corpus)

print(
    f"\nVocabulary Size (From Scratch): "
    f"{len(vocab_scratch)} words"
)

print(
    f"TF-IDF Matrix Shape: "
    f"{tfidf_scratch.shape}"
)


Vocabulary Size (From Scratch): 32 words
TF-IDF Matrix Shape: (3, 32)


In [29]:
# 1. From-Scratch Verification

vocab_scratch, tfidf_scratch = tfidf_from_scratch(corpus)

print(
    f"\nVocabulary Size (From Scratch): "
    f"{len(vocab_scratch)} words"
)

print(
    f"TF-IDF Matrix Shape: "
    f"{tfidf_scratch.shape}"
)


Vocabulary Size (From Scratch): 32 words
TF-IDF Matrix Shape: (3, 32)


In [30]:
print("\n--- TF-IDF Feature Values ---")

for i, document in enumerate(doc_ids):
    print(f"\nDocument: {document}")

    for j, word in enumerate(vocab_scratch):
        value = tfidf_scratch[i, j]

        if value > 0:
            print(
                f"{word:<20} : "
                f"{value:.4f}"
            )


--- TF-IDF Feature Values ---

Document: paper_1.txt
and                  : 0.1841
architectures        : 0.2371
convolutional        : 0.2371
deep                 : 0.2371
have                 : 0.2371
image                : 0.3117
language             : 0.2371
learning             : 0.2371
natural              : 0.2371
networks.            : 0.3117
neural               : 0.2371
processing           : 0.2371
recognition          : 0.3117
revolutionized       : 0.2371
tasks                : 0.2371
using                : 0.2371

Document: submission_A.txt
and                  : 0.1938
architectures        : 0.2495
convolutional        : 0.2495
deep                 : 0.2495
have                 : 0.2495
language             : 0.2495
layers.              : 0.3281
learning             : 0.2495
natural              : 0.2495
networks             : 0.3281
neural               : 0.2495
processing           : 0.2495
revolutionized       : 0.2495
tasks                : 0.2495
using             

In [31]:
# 2. Scikit-learn Comparison

sklearn_vec = TfidfVectorizer()

tfidf_sklearn = sklearn_vec.fit_transform(
    corpus
).toarray()

print(
    f"Scikit-Learn TF-IDF Matrix Shape: "
    f"{tfidf_sklearn.shape}"
)

Scikit-Learn TF-IDF Matrix Shape: (3, 31)


In [32]:
print("\n--- Scikit-Learn TF-IDF Matrix ---")
print(tfidf_sklearn)


--- Scikit-Learn TF-IDF Matrix ---
[[0.18801404 0.2421023  0.         0.         0.         0.
  0.2421023  0.2421023  0.         0.2421023  0.31833543 0.2421023
  0.         0.2421023  0.         0.         0.2421023  0.2421023
  0.2421023  0.         0.         0.2421023  0.         0.31833543
  0.2421023  0.         0.         0.2421023  0.         0.2421023
  0.        ]
 [0.19833162 0.25538807 0.         0.         0.         0.
  0.25538807 0.25538807 0.         0.25538807 0.         0.25538807
  0.33580462 0.25538807 0.         0.         0.25538807 0.25538807
  0.25538807 0.         0.         0.25538807 0.         0.
  0.25538807 0.         0.         0.25538807 0.         0.25538807
  0.        ]
 [0.14179804 0.         0.24008495 0.24008495 0.24008495 0.24008495
  0.         0.         0.24008495 0.         0.         0.
  0.         0.         0.24008495 0.24008495 0.         0.
  0.         0.24008495 0.24008495 0.         0.4801699  0.
  0.         0.24008495 0.24008495 

In [33]:
print("\n--- Scikit-Learn TF-IDF Matrix ---")
print(tfidf_sklearn)


--- Scikit-Learn TF-IDF Matrix ---
[[0.18801404 0.2421023  0.         0.         0.         0.
  0.2421023  0.2421023  0.         0.2421023  0.31833543 0.2421023
  0.         0.2421023  0.         0.         0.2421023  0.2421023
  0.2421023  0.         0.         0.2421023  0.         0.31833543
  0.2421023  0.         0.         0.2421023  0.         0.2421023
  0.        ]
 [0.19833162 0.25538807 0.         0.         0.         0.
  0.25538807 0.25538807 0.         0.25538807 0.         0.25538807
  0.33580462 0.25538807 0.         0.         0.25538807 0.25538807
  0.25538807 0.         0.         0.25538807 0.         0.
  0.25538807 0.         0.         0.25538807 0.         0.25538807
  0.        ]
 [0.14179804 0.         0.24008495 0.24008495 0.24008495 0.24008495
  0.         0.         0.24008495 0.         0.         0.
  0.         0.         0.24008495 0.24008495 0.         0.
  0.         0.24008495 0.24008495 0.         0.4801699  0.
  0.         0.24008495 0.24008495 

In [34]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    corpus
).toarray()

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("\n--- Document Similarity Matrix ---")
print(similarity_matrix)


--- Document Similarity Matrix ---
[[1.         0.84107963 0.02666002]
 [0.84107963 1.         0.02812303]
 [0.02666002 0.02812303 1.        ]]


In [35]:
print("\n--- Pairwise Document Similarity ---")

for i in range(len(doc_ids)):
    for j in range(i + 1, len(doc_ids)):
        
        score = similarity_matrix[i, j]
        
        print(
            f"{doc_ids[i]:<20} | "
            f"{doc_ids[j]:<20} | "
            f"Cosine Similarity: {score:.4f}"
        )


--- Pairwise Document Similarity ---
paper_1.txt          | submission_A.txt     | Cosine Similarity: 0.8411
paper_1.txt          | paper_2.txt          | Cosine Similarity: 0.0267
submission_A.txt     | paper_2.txt          | Cosine Similarity: 0.0281
